In [1]:
import os

import matplotlib.pyplot as plt
import matplotlib
import numpy

from astropy.coordinates import SkyCoord
import astropy.units as u

from rascil.processing_components import plot_uvcoverage, plot_visibility

from radioimaging.util import util
from radioimaging.visibility import vissim, residual, weights, bda, ingest
from radioimaging.images import images
from rascil.processing_components import export_visibility_to_ms

plt.set_loglevel("critical")
cmap='turbo'
matplotlib.rcParams['figure.figsize'] = [5, 5]

In [2]:
gt_name = "/home/sunwang/RIDeisa/data/galfield/galfield_gt.fits"
config_name = "MID"

phasecentre = SkyCoord(ra=+56.0 * u.deg, dec=-15.0 * u.deg, frame='icrs', equinox='J2000')
ha_interval=(-0.5,0.5)

bandwidth = 1e4
base_freq = 1e9
nchan = 1
freqs = range(int(base_freq), int(base_freq + bandwidth*nchan), int(bandwidth))
bandwidths = [bandwidth] * len(freqs)
int_time = 5

noise_level = 10
num_pix = util.fromfits(gt_name).shape[0]

tmp_ms_names = ["tmp/tmp" + str(i) + ".ms" for i in range(len(freqs))]
non_averaged_names = ["../data8/galfield" + str(i) + ".ms" for i in range(len(freqs))]

decorr_limit = 1

output_dataset_name = "../data8/mid.ms"

In [3]:
for i, freq in enumerate(freqs):
    vcurr = vissim.generate_visibilities(phasecentre,ha_interval,tel=config_name, integration_time=int_time, frequencies=numpy.array([freq]).astype(float), channel_bandwidths=numpy.array([bandwidth]))
    vcurr, cell_size, im = residual.visibilities_from_image(vcurr, gt_name,return_cellsize=True,return_image=True,scale_factor=1.8)
    print(vcurr.vis.shape)
    vcurr = vissim.add_noise_to_vis(vcurr, noise_level)
    print(cell_size)
    if decorr_limit < 1:
        cell_size_deg = cell_size * 180 / numpy.pi
        field_size_deg = cell_size_deg * num_pix 
        bda.bda_and_export_to_ms2(tmp_ms_names[i], vcurr, decorr_limit, field_size_deg / 2, only_residual_decorr=False)
    else:
        export_visibility_to_ms(non_averaged_names[i], [vcurr])

/home/sunwang/radio-imaging/environment/radioimaging/lib/python3.10/site-packages/ska_sdp_datamodels/image/image_model.py:101: RuntimeWarning: cdelt will be ignored since cd is present
  cellsize = numpy.deg2rad(numpy.abs(wcs.wcs.cdelt[1]))
/home/sunwang/radio-imaging/environment/radioimaging/lib/python3.10/site-packages/ska_sdp_datamodels/image/image_model.py:146: RuntimeWarning: cdelt will be ignored since cd is present
  "channel_bandwidth": wcs.wcs.cdelt[3],


(720, 19503, 1, 1)
8.755168963737264e-07


/home/sunwang/radio-imaging/environment/radioimaging/lib/python3.10/site-packages/ska_sdp_datamodels/image/image_model.py:101: RuntimeWarning: cdelt will be ignored since cd is present
  cellsize = numpy.deg2rad(numpy.abs(wcs.wcs.cdelt[1]))
/home/sunwang/radio-imaging/environment/radioimaging/lib/python3.10/site-packages/ska_sdp_datamodels/image/image_model.py:146: RuntimeWarning: cdelt will be ignored since cd is present
  "channel_bandwidth": wcs.wcs.cdelt[3],


(720, 19503, 1, 1)
8.755081412923134e-07


KeyboardInterrupt: 

In [ ]:
datasets_str = ""
for tms in tmp_ms_names:
    datasets_str += tms + " "

join_exec = "python join_datasets.py --idatasets " + datasets_str + "--odataset_name " + output_dataset_name

os.system(join_exec)

In [ ]:
for tms in tmp_ms_names:
    os.system("rm -r " + tms)